# Tuned SVM and RF

Hyperparameters are selected for every split by a grid search over the same cluster-wise inner folds as the epoch selection of the deep models, so the test clusters are never used. The SVM grid is C ∈ {0.1, 1, 10, 100} × γ ∈ {0.1, 1, 10}·γ0 and the RF grid is max_features ∈ {sqrt, log2, 0.05} × min_samples_leaf ∈ {1, 5, 20} (`tuned_core.py`). An interrupted run continues where it stopped.

In [ ]:
import os
PROTOCOLS = ["lopo", "l2po"]
MODELS = ["svm", "rf"]
SEARCH_STRIDE = 4
RUN_SHIFT = True
TIMING_ONLY = False
N_JOBS = os.cpu_count()
RESUME_FROM = None
TIME_BUDGET_H = 11.0
import time
T_START = time.time()
out_of_time = lambda: (time.time() - T_START) / 3600 > TIME_BUDGET_H

In [ ]:
import glob, json, shutil, sys, time
import numpy as np

ON_KAGGLE = os.path.isdir("/kaggle/input")
if ON_KAGGLE:
    def find(pattern):
        hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))
        assert hits, f"not found under /kaggle/input: {pattern}"
        return hits[0]
    PV2 = os.path.dirname(os.path.dirname(find("robustness/common.py")))
    MODULE = os.path.dirname(find("tuned_core.py"))
    RESULTS_DIR = "/kaggle/working/tuned_results"
else:
    MODULE = os.path.abspath(".")
    assert os.path.exists(os.path.join(MODULE, "tuned_core.py")), "run this notebook from Tuned_SVM_RF"
    PV2 = os.path.dirname(MODULE)
    RESULTS_DIR = os.path.join(MODULE, "results")
PATHS = (os.path.join(PV2, "robustness"), os.path.join(PV2, "CNN_LSTM", "cnn_lstm_core.py"))
sys.path.insert(0, MODULE)
import tuned_core as tc
tc.setup(*PATHS)
tc.SEARCH_STRIDE = SEARCH_STRIDE
cm = tc.cm
DATA_DIR = cm.ORIG_DATA_DIR
SHIFT_DIR = cm.SNR_DATA_DIR if RUN_SHIFT else None
os.makedirs(RESULTS_DIR, exist_ok=True)

if RESUME_FROM is None and ON_KAGGLE:
    old = glob.glob("/kaggle/input/**/tuned_results/**/*.jsonl", recursive=True)
    RESUME_FROM = (old[0].split("/tuned_results/")[0] + "/tuned_results") if old else None
if RESUME_FROM:
    for src in glob.glob(os.path.join(RESUME_FROM, "**", "*.jsonl"), recursive=True):
        dst = os.path.join(RESULTS_DIR, os.path.relpath(src, RESUME_FROM))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(src, dst)
    print("resumed from", RESUME_FROM)
print("PV2        ", PV2)
print("data       ", DATA_DIR)
print("SNR data   ", SHIFT_DIR)
print("results    ", RESULTS_DIR)
print("CPU cores  ", os.cpu_count())

## Step 1: Sanity checks

Splits and test-frame counts must equal those of the stored SVM and RF results.

In [ ]:
STORED = {}
for m, d in (("svm", "SVM"), ("rf", "RF")):
    for p in PROTOCOLS:
        rows = [json.loads(l) for l in open(os.path.join(PV2, d, "results", f"splits_{p}.jsonl")) if l.strip()]
        names = [s["name"] for s in cm.make_splits(p)]
        assert sorted(r["name"] for r in rows) == sorted(names), f"{d} {p}: split names differ"
        STORED[m, p] = {r["name"]: r for r in rows}
X, y = tc.data(DATA_DIR)
for p in PROTOCOLS:
    for s in cm.make_splits(p):
        n_test = sum(len(X[c]) for c in s["test"])
        assert n_test == STORED["svm", p][s["name"]]["n_test"] == STORED["rf", p][s["name"]]["n_test"]
        for f in tc.inner_folds(s["train"]):
            assert not set(f["train"] + f["val"]) & set(s["test"])
n_folds = {p: sum(len(tc.inner_folds(s["train"])) for s in cm.make_splits(p)) for p in PROTOCOLS}
print("frames per cluster:", {c: len(X[c]) for c in cm.ALL_CLUSTERS})
print("inner folds:", n_folds)
for m in MODELS:
    print(f"{m} grid ({len(tc.GRIDS[m])} points, first = default):", tc.GRIDS[m])
    print("   fixed:", tc.FIXED[m])
print("search tasks:", {m: len(tc.GRIDS[m]) * sum(n_folds.values()) for m in MODELS})

## Step 2: Timing

In [ ]:
from joblib import Parallel, delayed
split0 = cm.make_splits("lopo")[0]
est = 0.0
for m in MODELS:
    ts = []
    for g in range(len(tc.GRIDS[m])):
        r = tc.search_task(PATHS, DATA_DIR, m, "lopo", split0, 0, g, SEARCH_STRIDE)
        ts.append(r["seconds"])
        print(f"  {m} {str(r['params']):55s} balanced acc {r['balanced_accuracy']:.4f}   {r['seconds']:6.1f} s")
    search = sum(ts) * sum(n_folds.values())
    final = np.mean([STORED[m, p][n]["fit_seconds"] for p in PROTOCOLS for n in STORED[m, p]]) * 80 * 3
    print(f"{m}: search about {search / N_JOBS / 3600:.1f} h, final models about {final / N_JOBS / 3600:.1f} h "
          f"(with {N_JOBS} parallel jobs)")
    est += (search + final) / N_JOBS
print(f"\nestimate for everything: about {est / 3600:.1f} h (L2PO folds are smaller, so probably less)")
if TIMING_ONLY:
    raise SystemExit("TIMING_ONLY = True: stopping here")

## Step 3: Search

Finished tasks are appended to `search_{model}_{protocol}.jsonl` and skipped when the notebook runs again.

In [ ]:
from joblib import Parallel, delayed

def load_rows(path):
    rows, broken = [], 0
    if os.path.exists(path):
        for l in open(path):
            if not l.strip():
                continue
            try:
                rows.append(json.loads(l))
            except json.JSONDecodeError:
                broken += 1
        if broken:
            with open(path, "w") as f:
                f.writelines(json.dumps(r) + "\n" for r in rows)
            print(f"  {os.path.basename(path)}: dropped {broken} incomplete line(s)")
    return rows

def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

STOPPED = False

t_all = time.time()
for m in MODELS:
    for p in PROTOCOLS:
        path = os.path.join(RESULTS_DIR, f"search_{m}_{p}.jsonl")
        done = {r["key"] for r in load_rows(path)}
        tasks = [(s, k, g) for s in cm.make_splits(p) for k in range(len(tc.inner_folds(s["train"])))
                 for g in range(len(tc.GRIDS[m])) if f"{s['name']}|{k}|{g}" not in done]
        print(f"=== search {m} {cm.PROTOCOLS[p]}: {len(done)} done, {len(tasks)} to run", flush=True)
        if not tasks:
            continue
        t0, i = time.time(), 0
        with Parallel(n_jobs=N_JOBS, return_as="generator_unordered") as par:
            for part in chunks(tasks, N_JOBS * 8):
                if out_of_time():
                    STOPPED = True
                    break
                for r in par(delayed(tc.search_task)(PATHS, DATA_DIR, m, p, s, k, g, SEARCH_STRIDE) for s, k, g in part):
                    cm.append_result(path, r)
                    i += 1
                    if i % 50 == 0 or i == len(tasks):
                        print(f"  {i:5d}/{len(tasks)}  ({time.time() - t0:.0f} s)", flush=True)
        if STOPPED:
            break
    if STOPPED:
        break
print(f"search {'STOPPED (time budget)' if STOPPED else 'finished'} after {(time.time() - t_all) / 60:.0f} min")

## Step 4: Final models

The setting with the highest mean balanced accuracy over the inner folds is trained on all training frames and tested on the held-out clusters, also with the SNR-equalised LOS test clusters (condition B).

In [ ]:
t_all = time.time()
for m in ([] if STOPPED else MODELS):
    for p in PROTOCOLS:
        search = load_rows(os.path.join(RESULTS_DIR, f"search_{m}_{p}.jsonl"))
        path = os.path.join(RESULTS_DIR, m, f"splits_{p}.jsonl")
        done = {r["name"] for r in load_rows(path)}
        todo = []
        for s in cm.make_splits(p):
            if s["name"] in done:
                continue
            best, means = tc.select(m, s, [r for r in search if r["split"] == s["name"]])
            todo.append((s, best, means))
        print(f"=== final {m} {cm.PROTOCOLS[p]}: {len(done)} done, {len(todo)} to run")
        if not todo:
            continue
        t0, i = time.time(), 0
        with Parallel(n_jobs=N_JOBS, return_as="generator_unordered") as par:
            for part in chunks(todo, N_JOBS):
                if out_of_time():
                    STOPPED = True
                    break
                jobs = (delayed(tc.final_task)(PATHS, DATA_DIR, SHIFT_DIR, m, p, s, best, means, STORED[m, p][s["name"]])
                        for s, best, means in part)
                for r in par(jobs):
                    cm.append_result(path, r)
                    i += 1
                    st = r["stored_default"]["accuracy"]
                    shift = f"  condition B {r['test_shift']['accuracy']:.4f}" if "test_shift" in r else ""
                    print(f"  [{i:2d}/{len(todo)}] {r['name']:32s} {str(r['params']):50s} acc {r['accuracy']:.4f} "
                          f"(default {st:.4f}){shift}  ({time.time() - t0:.0f} s)", flush=True)
        if STOPPED:
            break
    if STOPPED:
        break
print(f"final models {'STOPPED (time budget)' if STOPPED else 'finished'} after {(time.time() - t_all) / 60:.0f} min")

## Step 5: Summary

Where the default setting is selected, the tuned result must equal the stored one.

In [ ]:
if STOPPED:
    print("NOT COMPLETE: the time budget was reached. Run the notebook again with this output attached as an input; "
          "finished tasks are skipped.")
summary = {"complete": not STOPPED, "grids": tc.GRIDS, "fixed": tc.FIXED, "search_stride": SEARCH_STRIDE,
           "criterion": "balanced accuracy on the held-out inner clusters, mean over the folds", "results": {}}
print(f"{'model':6s} {'protocol':12s} {'n':>3s} {'default':>8s} {'tuned':>8s} {'diff':>7s} {'B tuned':>8s}  default chosen")
for m in MODELS:
    for p in PROTOCOLS:
        rows = load_rows(os.path.join(RESULTS_DIR, m, f"splits_{p}.jsonl"))
        if not rows:
            continue
        acc = np.mean([r["accuracy"] for r in rows]) * 100
        dflt = np.mean([r["stored_default"]["accuracy"] for r in rows]) * 100
        n_def = sum(r["default_chosen"] for r in rows)
        same = [abs(r["accuracy"] - r["stored_default"]["accuracy"]) for r in rows if r["default_chosen"]]
        shift = np.mean([r["test_shift"]["accuracy"] for r in rows]) * 100 if "test_shift" in rows[0] else float("nan")
        from collections import Counter
        chosen = Counter(json.dumps(r["params"]) for r in rows).most_common()
        summary["results"][f"{m}/{p}"] = {"n": len(rows), "accuracy_tuned": acc, "accuracy_default": dflt,
                                          "test_shift_tuned": shift, "default_chosen": n_def,
                                          "max_diff_when_default": max(same) if same else None,
                                          "chosen": dict(chosen)}
        print(f"{m:6s} {cm.PROTOCOLS[p]:12s} {len(rows):3d} {dflt:8.2f} {acc:8.2f} {acc - dflt:+7.2f} {shift:8.2f}  "
              f"{n_def}/{len(rows)}")
        print("        chosen settings:", chosen)
        if same:
            print(f"        splits with the default setting: largest |tuned - stored| = {max(same):.2e}")
with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=1)
shutil.make_archive(os.path.join(os.path.dirname(RESULTS_DIR), "tuned_results"), "zip", RESULTS_DIR)
print("\nwritten:", os.path.join(RESULTS_DIR, "summary.json"), "and tuned_results.zip")